# OCR sobre crops del detector morfològic (Viola-Jones style)

Aquest notebook treballa sobre les imatges ja retallades per `01_morphologic_VJ.ipynb`,
guardades a `data/processed/` amb el patró `{nom_test}_box{n}.png`.

**Flux de treball:**
1. Per cada fitxer `_box{n}.png` a `data/processed/`, intentem llegir el GT des de `../data/raw/{test}.txt`
2. Si el GT existeix i el crop conté una matrícula (IoU o coincidència de bbox), apliquem OCR
3. Si el crop no correspon a cap matrícula real → el marquem com a fals positiu (no apliquem OCR)
4. Mesurem Accuracy, Precision i False Positive Rate del sistema complet


## 1. Imports i configuració de paths

In [ ]:
import cv2
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from collections import defaultdict
import re

# ── Paths ──────────────────────────────────────────────────────────────────
# Crops generats per 01_morphologic_VJ.ipynb
PROCESSED_DIR = Path('data/processed')

# Ground Truth: fitxers .txt amb format "img_name x y w h matricula"
RAW_DIR = Path('data/raw')

# Llindar IoU: un crop es considera "conté matrícula" si el seu solapament
# amb la bbox GT és >= IOU_THRESHOLD.
# Valor 0.0 → usem la bbox original del VJ tal qual (qualsevol solapament compta)
# Valor 0.5 → estàndard PASCAL VOC (recomanat)
IOU_THRESHOLD = 0.3

print(f'PROCESSED_DIR : {PROCESSED_DIR.resolve()}')
print(f'RAW_DIR       : {RAW_DIR.resolve()}')
print(f'IoU threshold : {IOU_THRESHOLD}')

## 2. Parsejat del Ground Truth

El fitxer `{test}.txt` conté **una línia** amb el format:
```
nom_imatge  x  y  w  h  MATRICULA
```
on `x, y, w, h` és la bounding box de la matrícula dins la imatge original.

Retornem la bbox com a `(x1, y1, x2, y2)` perquè sigui més fàcil calcular IoU.

In [ ]:
def parse_gt(txt_path: Path):
    """
    Llegeix el fitxer GT i retorna una llista de dicts:
      [{'img': str, 'x1':int, 'y1':int, 'x2':int, 'y2':int, 'plate': str}, ...]

    El format acceptat és flexible: separador de tabulació o espais.
    """
    entries = []
    if not txt_path.exists():
        return entries
    with open(txt_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            # Accepta tabulació o espais com a separador
            parts = line.split('\t') if '\t' in line else line.split()
            if len(parts) < 6:
                continue
            try:
                img_name = parts[0]
                x, y, w, h = int(parts[1]), int(parts[2]), int(parts[3]), int(parts[4])
                plate = parts[5].upper()
            except (ValueError, IndexError):
                continue
            if w > 0 and h > 0:
                entries.append({
                    'img': img_name,
                    'x1': x, 'y1': y,
                    'x2': x + w, 'y2': y + h,
                    'plate': plate
                })
    return entries


def iou(b1, b2):
    """
    Calcula la Intersection-over-Union entre dues bboxes (x1,y1,x2,y2).

    IoU = |A ∩ B| / |A ∪ B|

    Valors:
      1.0 → superposició perfecta
      0.0 → cap solapament
    """
    ix1 = max(b1['x1'], b2['x1'])
    iy1 = max(b1['y1'], b2['y1'])
    ix2 = min(b1['x2'], b2['x2'])
    iy2 = min(b1['y2'], b2['y2'])

    inter_w = max(0, ix2 - ix1)
    inter_h = max(0, iy2 - iy1)
    inter = inter_w * inter_h

    area1 = (b1['x2'] - b1['x1']) * (b1['y2'] - b1['y1'])
    area2 = (b2['x2'] - b2['x1']) * (b2['y2'] - b2['y1'])
    union = area1 + area2 - inter

    return inter / union if union > 0 else 0.0


print('Funcions parse_gt i iou definides ✓')

## 3. Extracció del nom de test i índex de box des del nom del fitxer

Els fitxers guardats per VJ segueixen el patró:
```
{stem}_box{n}.png
```
on `stem` és el nom de la imatge original sense extensió (ex: `test_001`, `eu3`...).

Per recuperar el GT, necessitem:
- El `stem` → `../data/raw/{stem}.txt`
- L'índex `n` → per saber quina box concreta (si n'hi ha múltiples) va generar el crop

**Però atenció**: el VJ guarda **totes les candidates** (incloent falsos positius).
Per saber si un crop conté realment una matrícula, comparem la bbox del crop
amb la bbox GT usant IoU.

In [ ]:
def parse_crop_filename(png_path: Path):
    """
    Extreu (stem, box_idx) del nom d'un fitxer de crop.

    Exemple: 'test_001_box2.png' → ('test_001', 2)

    Retorna (None, None) si el nom no segueix el patró.
    """
    name = png_path.stem  # sense extensió
    # Regex: captura tot el que hi ha ABANS de '_box' i el número
    m = re.match(r'^(.+)_box(\d+)$', name)
    if not m:
        return None, None
    stem = m.group(1)
    box_idx = int(m.group(2))
    return stem, box_idx


# Test ràpid
for test_name in ['test_001_box0.png', 'eu3_box2.png', 'im_with_underscore_name_box10.png']:
    s, i = parse_crop_filename(Path(test_name))
    print(f'  {test_name:35s} → stem={s!r:25s}  idx={i}')

## 4. Construcció de la base de dades de referència de caràcters

Igual que al notebook original, construïm el diccionari `ref_chars` a partir
dels fitxers GT de `data/raw/`. La diferència és que ara **llegim directament
les imatges originals** per fer el crop (no els crops del VJ), perquè volem que
la referència sigui neta i amb la bbox exacta del GT.

Usem els **primers 80% dels fitxers** per a referència i deixem la resta per a test.

In [ ]:
# ── Reutilitzem totes les funcions OCR del notebook original ───────────────

def preprocess_plate(crop):
    """Passa el crop a escala de grisos, suavitza i estira el contrast."""
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    gray = cv2.medianBlur(gray, 3)
    min_val, max_val = np.min(gray), np.max(gray)
    if max_val > min_val:
        gray = ((gray - min_val) / (max_val - min_val) * 255).astype(np.uint8)
    gray = cv2.resize(gray, None, fx=2.5, fy=2.5, interpolation=cv2.INTER_CUBIC)
    gray = cv2.GaussianBlur(gray, (3, 3), 0)
    return gray


def binarize(gray):
    """Binarització amb Otsu + assegurem que els caràcters siguin BLANCS."""
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if np.sum(binary == 255) < np.sum(binary == 0):
        binary = cv2.bitwise_not(binary)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=1)
    return binary


def smooth_projection(proj, k=9):
    proj = proj.astype(np.float32)
    kernel = np.ones((1, k), dtype=np.float32) / k
    return cv2.filter2D(proj.reshape(1, -1), -1, kernel).flatten()


def find_local_valley(proj, left, right, threshold):
    local = proj[left:right + 1]
    idx = int(np.argmin(local))
    return left + idx


def adjust_segments(segments, proj, expected_count):
    """Força que el número de segments sigui exactament expected_count."""
    if not segments:
        return segments
    segments = segments[:]
    avg_w = (segments[-1][1] - segments[0][0]) / max(expected_count, 1)

    # Dividir segments massa amples
    while len(segments) < expected_count:
        idx, (x1, x2) = max(enumerate(segments), key=lambda s: s[1][1] - s[1][0])
        width = x2 - x1
        if width < max(8, int(avg_w * 0.8)):
            break
        left  = x1 + max(int(width * 0.2), 1)
        right = x2 - max(int(width * 0.2), 1)
        if right <= left:
            break
        split = left + int(np.argmin(proj[left:right + 1]))
        if split <= x1 + 1 or split >= x2 - 1:
            break
        segments = segments[:idx] + [(x1, split), (split, x2)] + segments[idx + 1:]

    # Fusionar segments massa propers
    while len(segments) > expected_count:
        gaps = [(i, segments[i+1][0] - segments[i][1]) for i in range(len(segments) - 1)]
        i, _ = min(gaps, key=lambda t: t[1])
        segments = segments[:i] + [(segments[i][0], segments[i+1][1])] + segments[i+2:]

    return segments


def segment_by_projection(gray, expected_count):
    """Segmenta en N caràcters usant projecció vertical + ajust local."""
    binary = binarize(gray)
    proj = np.sum(binary == 255, axis=0)
    proj = smooth_projection(proj, k=9)
    valley_threshold = np.max(proj) * 0.25

    w = gray.shape[1]
    boundaries = [0]
    for i in range(1, expected_count):
        expected = int(i * w / expected_count)
        window = max(int(w * 0.05), 8)
        left  = max(0, expected - window)
        right = min(w - 1, expected + window)
        boundaries.append(find_local_valley(proj, left, right, valley_threshold))
    boundaries.append(w)

    segments = [(boundaries[i], boundaries[i+1])
                for i in range(len(boundaries) - 1)
                if boundaries[i+1] - boundaries[i] > 2]
    return adjust_segments(segments, proj, expected_count), binary, proj


def extract_char_images(gray, segments):
    binary = binarize(gray)
    chars = []
    for x1, x2 in segments:
        roi = binary[:, x1:x2]
        ys, xs = np.where(roi == 255)
        if len(xs) == 0:
            chars.append({'img': roi, 'bbox': (x1, 0, x2 - x1, roi.shape[0])})
            continue
        pad = 1
        y1b = max(0, ys.min() - pad);  y2b = min(roi.shape[0] - 1, ys.max() + pad)
        x1b = max(0, xs.min() - pad);  x2b = min(roi.shape[1] - 1, xs.max() + pad)
        char_img = roi[y1b:y2b + 1, x1b:x2b + 1]
        chars.append({'img': char_img, 'bbox': (x1 + x1b, y1b, x2b - x1b + 1, y2b - y1b + 1)})
    return chars


def normalize_char(img):
    if img is None or img.size == 0:
        return np.zeros((64, 32), dtype=np.uint8)
    h, w = img.shape
    scale = min(32 / max(w, 1), 64 / max(h, 1))
    new_w, new_h = max(1, int(w * scale)), max(1, int(h * scale))
    resized = cv2.resize(img, (new_w, new_h))
    canvas = np.zeros((64, 32), dtype=np.uint8)
    x0, y0 = (32 - new_w) // 2, (64 - new_h) // 2
    canvas[y0:y0 + new_h, x0:x0 + new_w] = resized
    return canvas


def edge_iou(a, b):
    inter = np.logical_and(a > 0, b > 0).sum()
    union = np.logical_or(a > 0, b > 0).sum()
    return inter / union if union > 0 else 0.0


def match_character(char_img, ref_chars, threshold=0.2):
    """Template matching sobre la base de referència. Retorna (char, score)."""
    char_norm = normalize_char(char_img)
    char_inv  = cv2.bitwise_not(char_norm)
    candidates = [
        (char_norm, cv2.Canny(char_norm, 40, 120)),
        (char_inv,  cv2.Canny(char_inv,  40, 120)),
    ]
    best_match, best_score = None, threshold
    for char_cand, char_edge in candidates:
        char_f = char_cand.astype(np.float32) / 255.0
        for ref_char, ref_images in ref_chars.items():
            for ref_item in ref_images:
                ref_norm = ref_item['img']
                ref_edge = ref_item['edge']
                ref_f    = ref_norm.astype(np.float32) / 255.0
                corr  = float(cv2.matchTemplate(char_f, ref_f, cv2.TM_CCOEFF_NORMED)[0][0])
                iou_e = edge_iou(char_edge, ref_edge)
                score = 0.7 * corr + 0.3 * iou_e
                if score > best_score:
                    best_score = score
                    best_match = ref_char
    return best_match, best_score


def ocr_plate(crop, gt_len, ref_chars):
    """Aplica OCR a un crop de matrícula. Retorna (text, chars_data, gray, binary, segments)."""
    gray = preprocess_plate(crop)
    segments, binary, proj = segment_by_projection(gray, gt_len)
    chars = extract_char_images(gray, segments)

    recognized = []
    for char_data in chars:
        matched, score = match_character(char_data['img'], ref_chars)
        if matched is None:
            matched, score = '?', 0.0
        recognized.append({
            'char': matched, 'score': score,
            'img': char_data['img'], 'bbox': char_data['bbox']
        })

    text = ''.join(r['char'] for r in recognized)
    return text, recognized, gray, binary, segments


print('Funcions OCR definides ✓')

## 5. Construcció de la base de referència de caràcters

Usem els GT de `data/raw/` per construir la referència. **Important**: la referència
es construeix a partir de les imatges *originals* (bbox GT exacta), no dels crops del VJ.
Això garanteix que la referència no conté soroll del detector.

In [ ]:
all_txt = sorted(RAW_DIR.glob('*.txt'))
print(f'Fitxers GT trobats: {len(all_txt)}')

# 80% per a referència, 20% per a test
n_ref = int(len(all_txt) * 0.8)
ref_txt_files  = all_txt[:n_ref]
test_txt_files = all_txt[n_ref:]

print(f'  Referència : {len(ref_txt_files)} fitxers')
print(f'  Test       : {len(test_txt_files)} fitxers')

# ── Construïm ref_chars ─────────────────────────────────────────────────
print('\nConstruint base de referència...')
ref_chars = defaultdict(list)
kernel_morph = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))

for txt_path in ref_txt_files:
    entries = parse_gt(txt_path)
    for entry in entries:
        img_path = RAW_DIR / entry['img']
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        x1, y1, x2, y2 = entry['x1'], entry['y1'], entry['x2'], entry['y2']
        crop = img[y1:y2, x1:x2]
        if crop.size == 0:
            continue
        gt = entry['plate']
        gray = preprocess_plate(crop)
        segments, _, _ = segment_by_projection(gray, len(gt))
        chars = extract_char_images(gray, segments)
        if len(chars) == len(gt):
            for char_data, gt_char in zip(chars, gt):
                base = normalize_char(char_data['img'])
                dil  = cv2.dilate(base, kernel_morph, iterations=1)
                ero  = cv2.erode(base,  kernel_morph, iterations=1)
                for variant in (base, dil, ero):
                    ref_chars[gt_char].append({
                        'img':  variant,
                        'edge': cv2.Canny(variant, 40, 120)
                    })

print('Caràcters de referència:')
for ch in sorted(ref_chars.keys()):
    print(f"  '{ch}': {len(ref_chars[ch])} mostres")

## 6. Descoberta dels crops del VJ i associació amb el GT

Per a cada `*_box*.png` de `data/processed/`:

1. Recuperem el `stem` i l'`idx` del nom del fitxer
2. Llegim el GT des de `data/raw/{stem}.txt`
3. **Reconstruïm la bbox del crop** dins la imatge original:
   - El VJ va guardar el fitxer `.png` a partir d'una bbox `(x, y, w, h)`
   - Per reconstruir-la, re-executem el detector (o assumim que tot el crop és la bbox)
   
> **Nota pedagògica**: el VJ no guarda quina era la bbox original de cada crop.
> Per saber si el crop conté la matrícula tenim dues estratègies:
> - **A (ràpida)**: mirem si la imatge GT del test té almenys 1 bbox detectada; si el test
>   té exactament 1 box detectada, asumim que és la matrícula.
> - **B (rigorosa)**: re-executem el detector per recuperar les coords originals i calculem IoU.
>
> Implementem l'estratègia B per màxim rigor.


In [ ]:
# ── Importem la pipeline de detecció del notebook morfològic ──────────────
# (Copiem les funcions necessàries per poder re-executar el detector)

def preprocess_vj(img_bgr):
    gray    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 1.0)
    clahe   = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(blurred)


def sobel_binary(gray):
    sobel_x   = cv2.Sobel(gray, cv2.CV_16S, 1, 0, ksize=3)
    sobel_abs = cv2.convertScaleAbs(sobel_x)
    _, binary = cv2.threshold(sobel_abs, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return binary


def morphology_label(binary):
    close_k = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 3))
    closed  = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, close_k)
    open_k  = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    morph   = cv2.morphologyEx(closed, cv2.MORPH_OPEN, open_k)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(morph, connectivity=8)
    return num_labels, labels, stats


# Paràmetres del filtre de forma (han de coincidir amb 01_morphologic_VJ.ipynb)
AREA_RATIO_MIN  = 0.001
AREA_RATIO_MAX  = 0.10
ASPECT_RATIO_MIN = 1.5
ASPECT_RATIO_MAX = 9.0
EXTENT_MIN       = 0.25
MIN_WIDTH        = 40
MIN_HEIGHT       = 10


def detect_boxes(img_bgr):
    """
    Re-executa el detector morfològic i retorna la llista de bboxes detectades,
    en el mateix ordre que el VJ les va numerar (idx 0, 1, 2...).
    Retorna: list of dict {'x1','y1','x2','y2'}
    """
    H, W = img_bgr.shape[:2]
    img_area = H * W
    enhanced = preprocess_vj(img_bgr)
    binary   = sobel_binary(enhanced)
    num_labels, _, stats = morphology_label(binary)

    boxes = []
    for lbl in range(1, num_labels):
        x, y, w, h, area = stats[lbl]
        if w < MIN_WIDTH or h < MIN_HEIGHT:
            continue
        if not (AREA_RATIO_MIN <= area / img_area <= AREA_RATIO_MAX):
            continue
        aspect = w / float(h)
        if not (ASPECT_RATIO_MIN <= aspect <= ASPECT_RATIO_MAX):
            continue
        extent = area / float(w * h)
        if extent < EXTENT_MIN:
            continue
        boxes.append({'x1': int(x), 'y1': int(y),
                      'x2': int(x + w), 'y2': int(y + h)})
    return boxes


print('Funció detect_boxes definida ✓')

## 7. Processament principal: OCR sobre crops del VJ

Per a cada crop del VJ:
- Si el crop **conté matrícula** (IoU ≥ llindar amb GT) → apliquem OCR i comparem amb GT
- Si el crop **no conté matrícula** (fals positiu del detector) → el registrem com a FP

Mètriques finals:
- **Plate Accuracy**: % de matrícules detectades i reconegudes correctament
- **Char Accuracy**: % de caràcters individuals correctes
- **False Positive Rate** del detector VJ

In [ ]:
crop_files = sorted(PROCESSED_DIR.glob('*_box*.png'))
print(f'Crops trobats a {PROCESSED_DIR}: {len(crop_files)}')

results = []

# Cache per evitar re-executar el detector múltiples vegades per la mateixa imatge
_detector_cache = {}  # stem → list of boxes

for crop_path in crop_files:
    stem, box_idx = parse_crop_filename(crop_path)
    if stem is None:
        print(f'  SKIP (nom inesperat): {crop_path.name}')
        continue

    # ── 1. Llegim el GT ────────────────────────────────────────────────────
    txt_path = RAW_DIR / f'{stem}.txt'
    gt_entries = parse_gt(txt_path)  # pot ser [] si el fitxer no existeix

    # ── 2. Carreguem la imatge original per re-detectar bboxes ─────────────
    # Necessitem la imatge original per saber la bbox exacta d'aquest crop
    # Intentem trobar-la (jpg, jpeg, png, bmp...)
    orig_img = None
    for ext in ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'):
        candidate = RAW_DIR / f'{stem}{ext}'
        if candidate.exists():
            orig_img = cv2.imread(str(candidate))
            break

    # ── 3. Recuperem la bbox d'aquest crop ────────────────────────────────
    crop_bbox = None  # dict amb x1,y1,x2,y2
    if orig_img is not None:
        if stem not in _detector_cache:
            _detector_cache[stem] = detect_boxes(orig_img)
        det_boxes = _detector_cache[stem]
        if box_idx < len(det_boxes):
            crop_bbox = det_boxes[box_idx]

    # ── 4. Decidim si el crop conté matrícula real ─────────────────────────
    is_plate  = False
    gt_plate  = None
    best_iou  = 0.0

    if crop_bbox is not None and gt_entries:
        # Calculem IoU amb cada GT d'aquest test (normalment n'hi ha 1)
        for gt_entry in gt_entries:
            gt_box = {'x1': gt_entry['x1'], 'y1': gt_entry['y1'],
                      'x2': gt_entry['x2'], 'y2': gt_entry['y2']}
            score = iou(crop_bbox, gt_box)
            if score > best_iou:
                best_iou = score
                if score >= IOU_THRESHOLD:
                    is_plate = True
                    gt_plate = gt_entry['plate']
    elif not gt_entries:
        # No hi ha GT → no hi ha matrícula a la imatge original
        is_plate = False

    # ── 5. Llegim el crop i apliquem OCR (si és matrícula) ────────────────
    crop_img = cv2.imread(str(crop_path))
    if crop_img is None:
        print(f'  ERROR llegint: {crop_path.name}')
        continue

    ocr_text      = None
    chars_data    = []
    gray_ocr      = None
    binary_ocr    = None
    segments_ocr  = []
    match         = False
    avg_score     = 0.0

    if is_plate and gt_plate:
        ocr_text, chars_data, gray_ocr, binary_ocr, segments_ocr = \
            ocr_plate(crop_img, len(gt_plate), ref_chars)
        match     = (ocr_text == gt_plate)
        avg_score = float(np.mean([c['score'] for c in chars_data])) if chars_data else 0.0

    results.append({
        'crop_path': crop_path,
        'stem':      stem,
        'box_idx':   box_idx,
        'is_plate':  is_plate,
        'best_iou':  best_iou,
        'gt':        gt_plate,
        'ocr':       ocr_text,
        'match':     match,
        'avg_score': avg_score,
        'chars_data': chars_data,
        'crop_img':  crop_img,
        'gray':      gray_ocr,
        'binary':    binary_ocr,
        'segments':  segments_ocr,
    })

    # Log per línia
    if is_plate:
        status = '✓' if match else '✗'
        print(f"{status} {crop_path.name:35s} IoU={best_iou:.2f}  "
              f"GT={gt_plate:10s}  OCR={ocr_text:10s}  score={avg_score:.2f}")
    else:
        print(f"  FP {crop_path.name:35s} IoU={best_iou:.2f}  (fals positiu del detector)")

print(f'\nTotal crops processats: {len(results)}')

## 8. Mètriques globals

Calculem tres mètriques diferenciades:

| Mètrica | Definició |
|---|---|
| **Plate Accuracy** | % de matrícules reconegudes sense cap error |
| **Char Accuracy** | % de caràcters individuals correctes |
| **FP Rate (detector)** | % de crops que NO contenien matrícula real |

In [ ]:
plate_results = [r for r in results if r['is_plate']]
fp_results    = [r for r in results if not r['is_plate']]

if plate_results:
    # ── Plate Accuracy ─────────────────────────────────────────────────────
    n_correct = sum(1 for r in plate_results if r['match'])
    plate_acc = n_correct / len(plate_results) * 100

    # ── Char Accuracy ──────────────────────────────────────────────────────
    total_chars   = 0
    correct_chars = 0
    for r in plate_results:
        if r['gt'] and r['ocr']:
            n = min(len(r['gt']), len(r['ocr']))
            correct_chars += sum(1 for a, b in zip(r['gt'], r['ocr']) if a == b)
            total_chars   += len(r['gt'])
    char_acc = (correct_chars / total_chars * 100) if total_chars > 0 else 0.0

    # ── False Positive Rate ────────────────────────────────────────────────
    fp_rate = len(fp_results) / len(results) * 100 if results else 0.0

    avg_score = np.mean([r['avg_score'] for r in plate_results])

    print('=' * 55)
    print('RESULTATS GLOBALS')
    print('=' * 55)
    print(f'  Crops totals         : {len(results)}')
    print(f'  Crops amb matrícula  : {len(plate_results)}')
    print(f'  Falsos positius (VJ) : {len(fp_results)} ({fp_rate:.1f}%)')
    print('-' * 55)
    print(f'  Plate Accuracy       : {n_correct}/{len(plate_results)} = {plate_acc:.1f}%')
    print(f'  Char Accuracy        : {correct_chars}/{total_chars} = {char_acc:.1f}%')
    print(f'  Score promedi OCR    : {avg_score:.3f}')
    print('=' * 55)
else:
    print('Cap crop identificat com a matrícula. Revisa IOU_THRESHOLD o el contingut de data/processed/')

## 9. Visualització dels resultats

### 9a. Matrícules detectades i reconegudes (encerts i errors)

In [ ]:
N_VIZ = 8  # nombre de mostres a visualitzar

for result in plate_results[:N_VIZ]:
    crop_img  = result['crop_img']
    chars_data = result['chars_data']
    binary    = result['binary']

    if not chars_data or binary is None:
        continue

    n_cols = len(chars_data) + 3  # crop + binary + info + caràcters
    fig = plt.figure(figsize=(max(12, n_cols * 2), 5))

    # Crop original
    ax1 = plt.subplot(2, n_cols, 1)
    ax1.imshow(cv2.cvtColor(crop_img, cv2.COLOR_BGR2RGB))
    ax1.set_title('Crop VJ')
    ax1.axis('off')

    # Imatge binaritzada
    ax2 = plt.subplot(2, n_cols, 2)
    ax2.imshow(binary, cmap='gray')
    ax2.set_title('Binary (OCR)')
    ax2.axis('off')

    # Panell d'informació
    color = 'green' if result['match'] else 'red'
    ax_info = plt.subplot(2, n_cols, 3)
    ax_info.axis('off')
    ax_info.text(0.05, 0.80, f"GT  : {result['gt']}",  fontsize=11, fontweight='bold')
    ax_info.text(0.05, 0.55, f"OCR : {result['ocr']}", fontsize=11, color=color, fontweight='bold')
    ax_info.text(0.05, 0.30, f"IoU : {result['best_iou']:.2f}", fontsize=9)
    ax_info.text(0.05, 0.10, f"Score: {result['avg_score']:.2f}", fontsize=9)

    # Caràcters individuals
    for i, char_data in enumerate(chars_data):
        ax = plt.subplot(2, n_cols, n_cols + 1 + i)
        char_viz = cv2.resize(char_data['img'], (32, 64))
        ax.imshow(char_viz, cmap='gray')
        ax.set_title(f"{char_data['char']}\n({char_data['score']:.2f})",
                     fontsize=9, fontweight='bold')
        ax.axis('off')

    status = '✓ CORRECTE' if result['match'] else '✗ ERROR'
    plt.suptitle(f"{result['crop_path'].name}  —  {status}",
                 fontsize=12, fontweight='bold',
                 color='green' if result['match'] else 'red')
    plt.tight_layout()
    plt.show()

### 9b. Falsos positius del detector VJ

In [ ]:
if fp_results:
    n_fp_show = min(8, len(fp_results))
    cols = 4
    rows = (n_fp_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(14, 4 * rows))
    axes = np.atleast_1d(axes).flatten()

    for ax, result in zip(axes, fp_results[:n_fp_show]):
        ax.imshow(cv2.cvtColor(result['crop_img'], cv2.COLOR_BGR2RGB))
        ax.set_title(f"{result['crop_path'].name}\nIoU={result['best_iou']:.2f} (FP)",
                     fontsize=8, color='orange')
        ax.axis('off')

    for ax in axes[n_fp_show:]:
        ax.axis('off')

    plt.suptitle(f'Falsos Positius del detector VJ ({len(fp_results)} total)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Cap fals positiu detectat (tots els crops contenen matrícula).')

### 9c. Taula resum de tots els resultats

In [ ]:
print(f"{'Fitxer crop':<40} {'GT':<12} {'OCR':<12} {'IoU':>5} {'Score':>6} {'Resultat'}")
print('-' * 90)
for r in results:
    if r['is_plate']:
        status = '✓' if r['match'] else '✗'
        print(f"{r['crop_path'].name:<40} {r['gt']:<12} {r['ocr']:<12} "
              f"{r['best_iou']:>5.2f} {r['avg_score']:>6.3f}  {status}")
    else:
        print(f"{r['crop_path'].name:<40} {'—':<12} {'—':<12} "
              f"{r['best_iou']:>5.2f} {'—':>6}  FP")